<a href="https://www.kaggle.com/code/bichotasomibola/finpro?scriptVersionId=373995537" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Reconnaissance d'avions militaires — détection, classification fine et recherche d'images similaires

**Module Pattern Recognition — Master 2 I2A — UMBB** · Notebook de projet (version restructurée et mise en conformité avec le cahier des charges)

**Problème.** À partir d'une photographie, localiser chaque avion militaire puis identifier son modèle exact parmi 34 classes visuellement proches (F-15/F-16/F-18/F-35, Rafale/EF2000/J-10…).

**Ce que ce notebook fait, et ce qu'il ne refait pas.**
Les deux réseaux finaux (détecteur YOLO11n et classifieur ResNet50 rééquilibré) ont été entraînés lors d'une exécution précédente ; leurs poids sont rechargés ici (dataset Kaggle « models »). Tout le reste est recalculé dans ce notebook : audit du jeu de données, baselines classiques, représentations apprises, méthodes d'ensemble, détection, recherche d'images (CBIR), évaluation et analyse des erreurs.

| Exigence du cahier des charges | Section |
|---|---|
| Problème réel, difficulté justifiée, jeu de données documenté | 1 |
| Split fixé une fois, sans fuite (audit des quasi-doublons) | 1.4 |
| Baseline classique solide (HOG, LBP, SVM, KNN, Random Forest) | 3 |
| Famille « représentations apprises » (ImageNet gelé, DINO auto-supervisé, autoencodeur, fine-tuning) | 4 |
| Famille « méthodes d'ensemble » (bagging/OOB, boosting, stacking hors-pli) | 5 |
| Famille « modules spécialisés » (détection d'objets + CBIR) | 6 |
| Évaluation rigoureuse : macro-F1, top-3, bout en bout | 6.2, 7 |
| Analyse d'erreurs, calibration, limites honnêtes | 7, 8 |
| Reproductibilité (graines, versions, résultats exportés) | 0, 9 |

**Trois protocoles d'évaluation, à ne pas confondre.**
- **A — recadrages annotés** : boîte annotée, redimensionnée en 224×224. Sert à comparer *toutes* les familles de méthodes à armes égales.
- **B — recadrages du détecteur** : boîtes prédites par YOLO appariées aux annotations (IoU ≥ 0,5). Conditionnel : les avions manqués sont exclus.
- **C — bout en bout** : images complètes de test (112 classes, dont 78 hors des 34), les avions manqués comptent comme erreurs.

**Discipline d'évaluation.** Choix d'hyperparamètres et de modèles sur la validation uniquement. Les prédictions de test sont calculées et stockées au fil du notebook mais ne sont lues qu'une fois, dans la section 7.1.

**Durée indicative (T4 + 4 vCPU).** Données/recadrages ≈ 20 min · baselines classiques ≈ 15 min · embeddings + DINO + autoencodeur ≈ 15 min · ensembles ≈ 20-30 min (CPU) · YOLO + pipeline ≈ 10 min · reste ≈ 10 min. Total ≈ 1 h 30 à 2 h. Les sections 3 et 5 n'utilisent pas le GPU : elles peuvent tourner dans une session CPU à partir des fichiers de cache pour économiser le quota.

## 0. Configuration et reproductibilité

Prérequis Kaggle : accélérateur GPU T4, Internet activé (installation d'Ultralytics et téléchargement des poids DINO), dataset *militaryaircraftdetectiondataset* ajouté, et un dataset privé contenant `yolo11n_best.pt` et `resnet50_balanced_best.pth`.

In [ ]:
!pip install -q ultralytics==8.4.216

In [ ]:
import os, glob, json, time, random, shutil, copy, warnings, platform
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------- Réglages ----------------
SEED = 42
MIN_IMAGES_PER_CLASS = 300      # seuil de sélection des classes (images uniques, tous splits)
CROP_SIZE = 224
N_PILOT_CLASSIC = 8000          # recadrages d'entraînement pour les baselines classiques (temps de calcul)
STACK_FOLDS = 3                 # plis du stacking hors-pli
NEAR_DUP_THR = 6                # distance de Hamming max (sur 64 bits) pour « quasi-doublon »
RUN_DEDUP, RUN_DINO, RUN_AE, RUN_YOLO_MAP = True, True, True, True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

# ---------------- Chemins ----------------
def find_first(patterns):
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    return None

LABELS_CSV = find_first([
    "/kaggle/input/datasets/a2015003713/militaryaircraftdetectiondataset/labels_with_split.csv",
    "/kaggle/input/*/labels_with_split.csv", "/kaggle/input/*/*/labels_with_split.csv",
    "/kaggle/input/*/*/*/labels_with_split.csv"])
assert LABELS_CSV, "labels_with_split.csv introuvable : ajoutez le dataset d'avions au notebook."
DATA_DIR = Path(LABELS_CSV).parent
IMG_DIR = DATA_DIR / "dataset"
if not IMG_DIR.exists():
    IMG_DIR = next(p for p in DATA_DIR.iterdir() if p.is_dir())
YOLO_W = find_first(["/kaggle/input/**/yolo11n_best.pt"])
RESNET_W = find_first(["/kaggle/input/**/resnet50_balanced_best.pth"])
assert YOLO_W and RESNET_W, "poids introuvables : ajoutez le dataset « models » (yolo11n_best.pt, resnet50_balanced_best.pth)."

OUT = Path("/kaggle/working"); CACHE = OUT / "cache"; FIG = OUT / "figures"
for d in (CACHE, FIG, CACHE / "crops" / "train", CACHE / "crops" / "validation", CACHE / "crops" / "test"):
    d.mkdir(parents=True, exist_ok=True)

print("Dataset :", DATA_DIR); print("Images  :", IMG_DIR); print("YOLO    :", YOLO_W); print("ResNet  :", RESNET_W)
print("Device  :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

In [ ]:
import sklearn, skimage, ultralytics
ENV = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
       "scikit-image": skimage.__version__, "ultralytics": ultralytics.__version__, "seed": SEED}
print(json.dumps(ENV, indent=2))

# ---------------- Utilitaires partagés ----------------
RESULTS = {}      # résultats de validation par modèle
TEST_PROBS = {}   # probabilités de test stockées, lues UNE fois en 7.1

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=140, bbox_inches="tight"); plt.show()

def metrics(y, pred, probs=None):
    d = {"accuracy": float(accuracy_score(y, pred)),
         "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0))}
    if probs is not None:
        top3 = np.argsort(-probs, axis=1)[:, :3]
        d["top3"] = float(np.mean([t in row for t, row in zip(y, top3)]))
    return d

def iou_matrix(A, B):
    A = np.asarray(A, float).reshape(-1, 4); B = np.asarray(B, float).reshape(-1, 4)
    x1 = np.maximum(A[:, None, 0], B[None, :, 0]); y1 = np.maximum(A[:, None, 1], B[None, :, 1])
    x2 = np.minimum(A[:, None, 2], B[None, :, 2]); y2 = np.minimum(A[:, None, 3], B[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aA = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1]); aB = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])
    return inter / (aA[:, None] + aB[None, :] - inter + 1e-9)

def greedy_match(gt, pred, thr=0.5):
    """Appariement un-à-un par IoU décroissante. Retourne {indice_gt: (indice_pred, iou)}."""
    if len(gt) == 0 or len(pred) == 0:
        return {}
    M = iou_matrix(gt, pred)
    cands = sorted(((M[i, j], i, j) for i, j in zip(*np.where(M >= thr))), reverse=True)
    ug, up, out = set(), set(), {}
    for v, i, j in cands:
        if i in ug or j in up:
            continue
        ug.add(i); up.add(j); out[i] = (j, float(v))
    return out

def ece_score(conf, correct, bins=15):
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

## 1. Problème et jeu de données

### 1.1 Chargement, nettoyage et sélection des classes
Jeu **Military Aircraft Detection Dataset** (Kaggle, T. Nakamura, 2020). **La licence n'est pas précisée et les images semblent provenir du web** : usage strictement académique, source citée, images non redistribuées.

Sélection : classes ayant au moins 300 images uniques (tous splits). Nettoyage : les boîtes sont tronquées aux dimensions de l'image et les boîtes dégénérées (< 2 px) sont écartées ; les images illisibles sont ignorées et comptées (l'ancienne version avait écarté 7 images à la main).

In [ ]:
df_raw = pd.read_csv(LABELS_CSV)
print(df_raw.shape); display(df_raw.head(3))

counts = df_raw.groupby("class")["filename"].nunique().sort_values(ascending=False)
CLASSES = sorted(counts[counts >= MIN_IMAGES_PER_CLASS].index)

# Correspondance avec le classifieur entraîné (classe -> indice)
ck = torch.load(RESNET_W, map_location="cpu", weights_only=False)
CLASS_TO_IDX = ck["class_to_idx"]
if sorted(CLASS_TO_IDX) != CLASSES:
    print("ATTENTION : classes recalculées ≠ classes du checkpoint ->", set(CLASSES) ^ set(CLASS_TO_IDX))
CLASSES = sorted(CLASS_TO_IDX, key=CLASS_TO_IDX.get)
N_CLASSES = len(CLASSES)

df = df_raw.copy()
for a, b in (("xmin", "width"), ("xmax", "width")):
    df[a] = df[a].clip(0, df[b])
for a, b in (("ymin", "height"), ("ymax", "height")):
    df[a] = df[a].clip(0, df[b])
bad = ((df.xmax - df.xmin) < 2) | ((df.ymax - df.ymin) < 2)
print(f"Boîtes dégénérées écartées : {int(bad.sum())}")
df = df[~bad].copy()
df["box_w"], df["box_h"] = df.xmax - df.xmin, df.ymax - df.ymin
df["area"] = df.box_w * df.box_h
df["supported"] = df["class"].isin(CLASSES)
sup_df = df[df.supported].copy()

summary = pd.DataFrame({
    "images": df.groupby("split").filename.nunique(),
    "boîtes (114 classes)": df.groupby("split").size(),
    "boîtes (34 classes)": sup_df.groupby("split").size()}).loc[["train", "validation", "test"]]
display(summary)
print(f"Classes d'origine : {df_raw['class'].nunique()} · classes retenues : {N_CLASSES} · "
      f"part des boîtes de test hors des 34 classes : {100 * (1 - df[df.split=='test'].supported.mean()):.1f} %")

### 1.2 Pourquoi ce problème est difficile
Déséquilibre des classes, tailles d'objets très variables, plusieurs avions par image, classes visuellement proches et présence de classes hors domaine.

In [ ]:
tr_counts = sup_df[sup_df.split == "train"]["class"].value_counts().reindex(CLASSES)
print(f"Déséquilibre (train, 34 classes) : min={tr_counts.min()} ({tr_counts.idxmin()}) · max={tr_counts.max()} ({tr_counts.idxmax()}) · rapport={tr_counts.max()/tr_counts.min():.2f}")
print("Avions par image : médiane", int(df.groupby("filename").size().median()), "· max", int(df.groupby("filename").size().max()))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
tr_counts.sort_values().plot.barh(ax=ax[0], color="#3FB8AF"); ax[0].set_title("Boîtes d'entraînement par classe (34 classes)")
ax[1].hist(np.log10(sup_df.area), bins=60, color="#E8A33D"); ax[1].set_xlabel("log10(surface de la boîte en px²)")
ax[1].set_title("Distribution des tailles d'objets")
savefig("fig01_donnees")

sample = sup_df[sup_df.split == "train"].drop_duplicates("filename").sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for a, (_, r) in zip(axes.ravel(), sample.iterrows()):
    im = Image.open(IMG_DIR / f"{r.filename}.jpg"); im.draft("RGB", (800, 800)); sc = im.width / r.width
    a.imshow(im); a.axis("off")
    for _, g in sup_df[sup_df.filename == r.filename].iterrows():
        a.add_patch(plt.Rectangle((g.xmin * sc, g.ymin * sc), g.box_w * sc, g.box_h * sc, fill=False, color="lime", lw=1.5))
    a.set_title(r["class"], fontsize=10)
savefig("fig02_exemples")

### 1.3 Intégrité du découpage
Le découpage train/validation/test est **celui du jeu de données, défini une seule fois au niveau de l'image** ; les deux réseaux finaux ont été entraînés dessus, on ne peut donc pas le refaire sans tout réentraîner.

In [ ]:
n_multi = int((df.groupby("filename").split.nunique() > 1).sum())
print("Images présentes dans plus d'un split :", n_multi)
assert n_multi == 0

### 1.4 Audit des quasi-doublons (fuite potentielle train → test)
Un hash perceptuel différentiel (dHash, 64 bits) est calculé pour chaque image ; deux images à distance de Hamming ≤ `NEAR_DUP_THR` sont considérées quasi identiques. On mesure :
- combien d'images de validation/test ont un quasi-doublon dans le train (fuite potentielle) ;
- des **groupes de quasi-doublons du train**, réutilisés plus loin pour une validation croisée *groupée* (ensembles, section 5).

Cet audit ne change pas le split : il permet de mesurer l'impact éventuel (section 7.5) sur le test « dédupliqué ».

In [ ]:
LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def dhash(path, size=8):
    try:
        im = Image.open(path); im.draft("L", (size * 16, size * 16))
        a = np.asarray(im.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
        return np.packbits((a[:, 1:] > a[:, :-1]).flatten())          # 8 octets = 64 bits
    except Exception:
        return None

def hamming_pairs(A, B, thr, same=False, chunk=256):
    out = []
    for s in range(0, len(A), chunk):
        d = LUT[np.bitwise_xor(A[s:s + chunk, None, :], B[None, :, :])].sum(-1, dtype=np.uint8)
        ii, jj = np.nonzero(d <= thr); ii = ii + s
        if same:
            k = ii < jj; ii, jj = ii[k], jj[k]
        out.append(np.stack([ii, jj], 1))
    return np.concatenate(out) if out else np.zeros((0, 2), int)

def components(n, pairs):
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in pairs:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    return np.array([find(i) for i in range(n)])

LEAK_TEST, LEAK_VAL, GROUP = set(), set(), {}
if RUN_DEDUP:
    files = np.array(sorted(sup_df.filename.unique()))
    with ThreadPoolExecutor(8) as ex:
        hs = list(tqdm(ex.map(lambda f: dhash(IMG_DIR / f"{f}.jpg"), files), total=len(files), desc="dHash"))
    ok = np.array([h is not None for h in hs])
    H = np.stack([h if h is not None else np.zeros(8, np.uint8) for h in hs])
    split_of = sup_df.drop_duplicates("filename").set_index("filename").split.reindex(files).values
    tr_i, va_i, te_i = [np.where((split_of == s) & ok)[0] for s in ("train", "validation", "test")]
    p_te = hamming_pairs(H[te_i], H[tr_i], NEAR_DUP_THR)
    p_va = hamming_pairs(H[va_i], H[tr_i], NEAR_DUP_THR)
    p_tr = hamming_pairs(H[tr_i], H[tr_i], NEAR_DUP_THR, same=True)
    LEAK_TEST = set(files[te_i[np.unique(p_te[:, 0])]]) if len(p_te) else set()
    LEAK_VAL = set(files[va_i[np.unique(p_va[:, 0])]]) if len(p_va) else set()
    comp = components(len(tr_i), p_tr)
    GROUP = dict(zip(files[tr_i], comp))
    print(f"Images illisibles : {int((~ok).sum())}")
    print(f"Test : {len(LEAK_TEST)}/{len(te_i)} images ({100*len(LEAK_TEST)/max(1,len(te_i)):.1f} %) ont un quasi-doublon dans le train")
    print(f"Validation : {len(LEAK_VAL)}/{len(va_i)} images ({100*len(LEAK_VAL)/max(1,len(va_i)):.1f} %)")
    print(f"Train : {len(np.unique(comp))} groupes pour {len(tr_i)} images ({int((np.bincount(comp)>1).sum())} groupes de quasi-doublons)")
else:
    print("Audit des quasi-doublons désactivé.")

## 2. Recadrages (protocole A)

Un recadrage = boîte annotée exacte, redimensionnée en 224×224 (bicubique, ratio non conservé : même prétraitement que celui du classifieur final). Les recadrages sont mis en cache sur disque ; chaque image n'est décodée qu'une fois.

In [ ]:
def _crop_job(item):
    fn, g = item
    try:
        im = Image.open(IMG_DIR / f"{fn}.jpg").convert("RGB")
    except Exception:
        return [(i, False) for i in g.index]
    res = []
    for i, r in g.iterrows():
        p = CACHE / "crops" / r.split / f"{i}.jpg"
        try:
            if not p.exists():
                box = (int(r.xmin), int(r.ymin), int(np.ceil(r.xmax)), int(np.ceil(r.ymax)))
                im.crop(box).resize((CROP_SIZE, CROP_SIZE), Image.BICUBIC).save(p, quality=95)
            res.append((i, True))
        except Exception:
            res.append((i, False))
    return res

t0 = time.time()
groups = list(sup_df.groupby("filename"))
with ThreadPoolExecutor(8) as ex:
    out = list(tqdm(ex.map(_crop_job, groups), total=len(groups), desc="recadrages"))
ok_map = dict(r for chunk in out for r in chunk)
crops_df = sup_df.loc[[i for i, ok in ok_map.items() if ok]].copy()
crops_df["crop_path"] = [str(CACHE / "crops" / s / f"{i}.jpg") for i, s in zip(crops_df.index, crops_df.split)]
crops_df["y"] = crops_df["class"].map(CLASS_TO_IDX)
crops_df["leak"] = crops_df.filename.isin(LEAK_TEST) | crops_df.filename.isin(LEAK_VAL)
crops_df = crops_df.reset_index(drop=True)
ID = {s: np.where(crops_df.split.values == s)[0] for s in ("train", "validation", "test")}
Y = crops_df.y.values.astype(int)
print(f"{len(crops_df)} recadrages en {(time.time()-t0)/60:.1f} min · échecs : {len(ok_map) - len(crops_df)}")
print({s: len(v) for s, v in ID.items()})

## 3. Baseline classique (descripteurs HOG et LBP + classifieurs usuels)

Choix : **HOG** (forme locale : contours d'ailes, dérives, fuselage) et **LBP uniforme** (texture/peinture), deux descripteurs classiques complémentaires, à 128×128 en niveaux de gris. Classifieurs : SVM linéaire (SGD, perte hinge), k plus proches voisins sur ACP, forêt aléatoire. Pour des raisons de temps de calcul, l'entraînement se fait sur un **sous-échantillon stratifié** de `N_PILOT_CLASSIC` recadrages ; les hyperparamètres sont choisis sur la validation (complète).

In [ ]:
from joblib import Parallel, delayed
from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

def feat_classic(path):
    g = np.asarray(Image.open(path).convert("L").resize((128, 128), Image.BILINEAR))
    h = hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")
    lbp = local_binary_pattern(g, 8, 1, "uniform")
    hist = [np.histogram(lbp[i*32:(i+1)*32, j*32:(j+1)*32], bins=10, range=(0, 10))[0] / 1024.0
            for i in range(4) for j in range(4)]
    return h.astype(np.float32), np.concatenate(hist).astype(np.float32)

pilot_tr = ID["train"]
if N_PILOT_CLASSIC < len(pilot_tr):
    pilot_tr, _ = train_test_split(pilot_tr, train_size=N_PILOT_CLASSIC, stratify=Y[pilot_tr], random_state=SEED)
sel = np.concatenate([pilot_tr, ID["validation"], ID["test"]])
t0 = time.time()
feats = Parallel(n_jobs=-1, batch_size=64)(delayed(feat_classic)(crops_df.crop_path[i]) for i in tqdm(sel))
HOG = np.stack([f[0] for f in feats]); LBP = np.stack([f[1] for f in feats])
pos = {i: k for k, i in enumerate(sel)}
ix = lambda ids: np.array([pos[i] for i in ids])
print(f"HOG {HOG.shape[1]} dim · LBP {LBP.shape[1]} dim · {(time.time()-t0)/60:.1f} min")

In [ ]:
sets = {"HOG": HOG, "LBP": LBP, "HOG+LBP": np.hstack([HOG, LBP * 10])}
ytr_c, yva_c, yte_c = Y[pilot_tr], Y[ID["validation"]], Y[ID["test"]]
rows, best = [], {"macro_f1": -1}

def try_model(name, fit_predict):
    global best
    t = time.time(); pv, pt, extra = fit_predict()
    m = metrics(yva_c, pv); m.update(model=name, seconds=round(time.time() - t, 1), **extra); rows.append(m)
    if m["macro_f1"] > best["macro_f1"]:
        best = {"macro_f1": m["macro_f1"], "model": name, "test_pred": pt}
    print(f'{name:38s} acc={m["accuracy"]:.3f} F1={m["macro_f1"]:.3f} ({m["seconds"]}s)')

def svm(Xn, alpha):
    def f():
        X = sets[Xn]; clf = make_pipeline(StandardScaler(with_mean=False), SGDClassifier(
            loss="hinge", alpha=alpha, class_weight="balanced", max_iter=100, tol=0.01, random_state=SEED))
        clf.fit(X[ix(pilot_tr)], ytr_c)
        return clf.predict(X[ix(ID["validation"])]), clf.predict(X[ix(ID["test"])]), {"param": f"alpha={alpha}"}
    return f

for Xn in sets:
    for alpha in (1e-5, 1e-4, 1e-3):
        try_model(f"SVM linéaire · {Xn} · α={alpha:g}", svm(Xn, alpha))

Xc = sets["HOG+LBP"]
pca_c = PCA(100, random_state=SEED).fit(Xc[ix(pilot_tr)])
Zc = pca_c.transform(Xc)
for k in (3, 5, 11):
    def f(k=k):
        clf = KNeighborsClassifier(k, weights="distance", n_jobs=-1).fit(Zc[ix(pilot_tr)], ytr_c)
        return clf.predict(Zc[ix(ID["validation"])]), clf.predict(Zc[ix(ID["test"])]), {"param": f"k={k}"}
    try_model(f"KNN · HOG+LBP (ACP 100) · k={k}", f)
def f():
    clf = RandomForestClassifier(300, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED).fit(Zc[ix(pilot_tr)], ytr_c)
    return clf.predict(Zc[ix(ID["validation"])]), clf.predict(Zc[ix(ID["test"])]), {"param": "300 arbres"}
try_model("Random Forest · HOG+LBP (ACP 100)", f)

classic_table = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)[["model", "accuracy", "macro_f1", "seconds"]]
display(classic_table.style.format({"accuracy": "{:.3f}", "macro_f1": "{:.3f}"}))
BEST_CLASSIC = best
print("Meilleure baseline classique (validation) :", best["model"])
TEST_PROBS["Baseline classique (meilleure sur val.)"] = ("pred_only", best["test_pred"])
RESULTS["Baseline classique (meilleure sur val.)"] = {"val": {"accuracy": float(classic_table.iloc[0].accuracy), "macro_f1": float(classic_table.iloc[0].macro_f1)}, "model": best["model"]}

## 4. Représentations apprises (famille 1)

Quatre représentations sont comparées **avec le même protocole A** (recadrages annotés, sonde linéaire entraînée sur le train, sélection sur la validation) :
1. **ResNet50 ImageNet gelé** (supervisé) — *linear probing* ;
2. **DINO ResNet50 gelé** (auto-supervisé, même architecture) — *linear probing* ;
3. **Autoencodeur convolutionnel** appris sur nos recadrages (64×64, latent 128) ;
4. **ResNet50 fine-tuné** sur les avions, avec échantillonnage rééquilibré (modèle final, poids rechargés).

Justification : comparer supervisé / auto-supervisé / non supervisé / fine-tuné isole ce qu'apporte l'adaptation au domaine (avions vus de loin, petits objets).

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
TFM = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class PathDS(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return TFM(Image.open(self.paths[i]).convert("RGB"))

@torch.inference_mode()
def embed(body, paths, fc=None, bs=128):
    """Retourne (features float16, logits float32 ou None) ; body: module image -> vecteur."""
    body.eval(); feats, logits = [], []
    for x in tqdm(DataLoader(PathDS(paths), batch_size=bs, num_workers=4), desc="embeddings"):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            f = body(x.to(DEVICE)).flatten(1)
        feats.append(f.float().cpu().half())
        if fc is not None:
            logits.append(fc(f.float()).float().cpu())
    return torch.cat(feats).numpy(), (torch.cat(logits).numpy() if fc is not None else None)

def cached(name, fn):
    p = CACHE / f"{name}.npz"
    if p.exists():
        z = np.load(p, allow_pickle=True); return {k: z[k] for k in z.files}
    r = fn(); np.savez(p, **r); return r

def linear_probe(Xtr, ytr, Xva, yva, epochs=60, lr=1e-3, wd=1e-4):
    """Sonde linéaire (couche dense) standardisée ; meilleur point sur le macro-F1 de validation."""
    torch.manual_seed(SEED)
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32); mu, sd = Xtr_t.mean(0), Xtr_t.std(0) + 1e-6
    Xtr_t = ((Xtr_t - mu) / sd).to(DEVICE); ytr_t = torch.tensor(ytr).to(DEVICE)
    Xva_t = ((torch.tensor(Xva, dtype=torch.float32) - mu) / sd).to(DEVICE)
    lin = nn.Linear(Xtr_t.shape[1], N_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(lin.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best_f1, best_state = -1, None
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr_t), device=DEVICE)
        for s in range(0, len(perm), 512):
            b = perm[s:s + 512]; opt.zero_grad()
            F.cross_entropy(lin(Xtr_t[b]), ytr_t[b]).backward(); opt.step()
        sched.step()
        with torch.no_grad():
            f1 = f1_score(yva, lin(Xva_t).argmax(1).cpu().numpy(), average="macro")
        if f1 > best_f1:
            best_f1, best_state = f1, copy.deepcopy(lin.state_dict())
    lin.load_state_dict(best_state)
    @torch.no_grad()
    def predict_proba(X):
        Z = ((torch.tensor(X, dtype=torch.float32) - mu) / sd).to(DEVICE)
        return torch.softmax(lin(Z), 1).cpu().numpy()
    return predict_proba

EMB = {}
PATHS = crops_df.crop_path.values

def probe_report(name, X):
    pp = linear_probe(X[ID["train"]], Y[ID["train"]], X[ID["validation"]], Y[ID["validation"]])
    pv = pp(X[ID["validation"]])
    RESULTS[name] = {"val": metrics(Y[ID["validation"]], pv.argmax(1), pv)}
    TEST_PROBS[name] = pp(X[ID["test"]])
    print(name, RESULTS[name]["val"])

### 4.1 ResNet50 ImageNet gelé (supervisé)

In [ ]:
def _imnet():
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(DEVICE)
    body = nn.Sequential(*list(m.children())[:-1])
    f, _ = embed(body, PATHS); return {"X": f}
EMB["imagenet"] = cached("emb_imagenet", _imnet)["X"].astype(np.float32)
probe_report("ResNet50 ImageNet gelé + sonde linéaire", EMB["imagenet"])

### 4.2 DINO ResNet50 gelé (auto-supervisé)
DINO (distillation sans étiquettes) apprend des représentations sans supervision ; même architecture que 4.1, donc comparaison directe supervisé / auto-supervisé. Nécessite Internet (poids via `torch.hub`).

In [ ]:
if RUN_DINO:
    try:
        def _dino():
            m = torch.hub.load("facebookresearch/dino:main", "dino_resnet50").to(DEVICE)
            f, _ = embed(m, PATHS); return {"X": f}
        EMB["dino"] = cached("emb_dino", _dino)["X"].astype(np.float32)
        probe_report("DINO ResNet50 gelé + sonde linéaire", EMB["dino"])
    except Exception as e:
        print("DINO indisponible (Internet ?) :", repr(e)[:200])

### 4.3 Autoencodeur convolutionnel (non supervisé)
Entraîné sur les recadrages du train uniquement (64×64), perte de reconstruction MSE ; le code latent (128 dimensions) sert de représentation.

In [ ]:
if RUN_AE:
    def _load64(p): return np.asarray(Image.open(p).convert("RGB").resize((64, 64), Image.BILINEAR))
    p64 = CACHE / "small64.npy"
    if p64.exists():
        SMALL = np.load(p64)
    else:
        with ThreadPoolExecutor(8) as ex:
            SMALL = np.stack(list(tqdm(ex.map(_load64, PATHS), total=len(PATHS), desc="64x64")))
        np.save(p64, SMALL)

    class AE(nn.Module):
        def __init__(s, z=128):
            super().__init__()
            blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
            up = lambda i, o: nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
            s.enc = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), blk(128, 256), nn.Flatten(), nn.Linear(256 * 16, z))
            s.fc = nn.Linear(z, 256 * 16)
            s.dec = nn.Sequential(up(256, 128), up(128, 64), up(64, 32), nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())
        def forward(s, x):
            z = s.enc(x); return s.dec(s.fc(z).view(-1, 256, 4, 4)), z

    to_t = lambda a: torch.tensor(a).permute(0, 3, 1, 2).float() / 255.0
    ae = AE().to(DEVICE); opt = torch.optim.Adam(ae.parameters(), 1e-3)
    Xtr_s = to_t(SMALL[ID["train"]]); losses = []
    for ep in range(15):
        ae.train(); perm = torch.randperm(len(Xtr_s)); tot = 0
        for s in range(0, len(perm), 256):
            x = Xtr_s[perm[s:s + 256]].to(DEVICE); opt.zero_grad()
            rec, _ = ae(x); loss = F.mse_loss(rec, x); loss.backward(); opt.step(); tot += loss.item() * len(x)
        losses.append(tot / len(perm))
    print("MSE de reconstruction (train), 1ère → dernière époque :", round(losses[0], 4), "→", round(losses[-1], 4))
    ae.eval(); Z = []
    with torch.no_grad():
        for s in range(0, len(SMALL), 512):
            Z.append(ae(to_t(SMALL[s:s + 512]).to(DEVICE))[1].cpu())
    EMB["ae"] = torch.cat(Z).numpy()
    probe_report("Autoencodeur (latent 128) + sonde linéaire", EMB["ae"])

    with torch.no_grad():
        x = to_t(SMALL[ID["test"][:8]]).to(DEVICE); rec = ae(x)[0].cpu()
    fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
    for k in range(8):
        ax[0, k].imshow(x[k].cpu().permute(1, 2, 0)); ax[1, k].imshow(rec[k].permute(1, 2, 0)); ax[0, k].axis("off"); ax[1, k].axis("off")
    ax[0, 0].set_title("original", loc="left"); ax[1, 0].set_title("reconstruit", loc="left")
    savefig("fig03_autoencodeur")

### 4.4 ResNet50 fine-tuné et rééquilibré (modèle final)

Entraînement (hors de ce notebook, journaux ci-dessous) : ResNet50 ImageNet, fine-tuning complet sur les recadrages du détecteur (AdamW 1e-4, 15 époques, symétrie horizontale), puis reprise avec `WeightedRandomSampler` (AdamW 3e-5, 10 époques, sélection sur macro-F1 de validation). Les poids rechargés ici sont ceux de la meilleure époque (époque 9).

⚠️ Le gain apporté par le rééquilibrage est **confondu** avec 10 époques d'entraînement supplémentaires à plus faible taux d'apprentissage (voir limites, section 8).

In [ ]:
ft = models.resnet50(weights=None); ft.fc = nn.Linear(2048, N_CLASSES)
ft.load_state_dict(ck["model_state_dict"]); ft.to(DEVICE).eval()
FT_BODY, FT_FC = nn.Sequential(*list(ft.children())[:-1]), ft.fc

def _ft():
    f, lg = embed(FT_BODY, PATHS, fc=FT_FC); return {"X": f, "logits": lg}
_e = cached("emb_finetuned", _ft)
EMB["finetuned"], FT_LOGITS = _e["X"].astype(np.float32), _e["logits"]
FT_PROBS = torch.softmax(torch.tensor(FT_LOGITS), 1).numpy()
RESULTS["ResNet50 fine-tuné rééquilibré (modèle final)"] = {"val": metrics(Y[ID["validation"]], FT_PROBS[ID["validation"]].argmax(1), FT_PROBS[ID["validation"]])}
TEST_PROBS["ResNet50 fine-tuné rééquilibré (modèle final)"] = FT_PROBS[ID["test"]]
print(RESULTS["ResNet50 fine-tuné rééquilibré (modèle final)"]["val"])

# Courbes d'apprentissage relevées dans les journaux de l'entraînement précédent
LOG_GT = dict(train=[64.26,88.99,97.30,98.95,99.20,98.61,98.84,99.05,99.16,99.00,99.29,99.84,99.95,99.95,99.87,99.92,99.85,99.90,99.90,99.94],
              val=[63.54,67.22,68.96,69.70,68.92,68.08,69.96,71.84,70.30,71.50,71.60,73.64,74.40,74.54,74.28,75.32,73.90,74.54,73.28,75.50])
LOG_YO = dict(train=[50.39,85.38,93.61,96.47,97.66,98.01,97.83,98.35,98.82,98.35,98.75,98.91,98.78,99.00,99.06],
              val=[75.39,82.30,83.68,85.04,83.81,83.99,85.28,84.47,83.31,85.44,85.85,85.40,85.90,83.61,84.36])
LOG_BA = dict(train=[99.74,99.88,99.89,99.90,99.95,99.96,100.0,99.97,99.95,99.97],
              val=[87.73,88.63,88.77,88.45,87.80,88.95,88.91,88.93,89.09,88.63])
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for a, (t, L) in zip(ax, [("Fine-tuning, recadrages annotés", LOG_GT), ("Fine-tuning, recadrages YOLO", LOG_YO), ("Reprise rééquilibrée", LOG_BA)]):
    a.plot(range(1, len(L["train"]) + 1), L["train"], label="train"); a.plot(range(1, len(L["val"]) + 1), L["val"], label="validation")
    a.set_title(t); a.set_xlabel("époque"); a.legend()
ax[0].set_ylabel("accuracy (%)"); savefig("fig04_courbes_apprentissage")

### 4.5 Visualisation des espaces latents (ACP et t-SNE)
Validation, 3 000 points tirés au hasard. Une bonne représentation regroupe les classes ; on s'attend à ce que le fine-tuning sépare nettement mieux que les caractéristiques gelées.

In [ ]:
from sklearn.manifold import TSNE
rs = np.random.RandomState(SEED)
sub = rs.choice(ID["validation"], size=min(3000, len(ID["validation"])), replace=False)
names_emb = [k for k in ("imagenet", "dino", "ae", "finetuned") if k in EMB]
titles = {"imagenet": "ResNet50 ImageNet gelé", "dino": "DINO ResNet50 gelé", "ae": "Autoencodeur", "finetuned": "ResNet50 fine-tuné"}
fig, axes = plt.subplots(2, len(names_emb), figsize=(4.6 * len(names_emb), 9))
axes = np.atleast_2d(axes).reshape(2, -1)
cmap = plt.get_cmap("nipy_spectral", N_CLASSES)
for j, k in enumerate(names_emb):
    Zp = PCA(2, random_state=SEED).fit_transform(EMB[k][sub])
    axes[0, j].scatter(Zp[:, 0], Zp[:, 1], c=Y[sub], cmap=cmap, s=4); axes[0, j].set_title(f"ACP — {titles[k]}")
    Zt = TSNE(2, init="pca", perplexity=30, random_state=SEED).fit_transform(PCA(50, random_state=SEED).fit_transform(EMB[k][sub]))
    axes[1, j].scatter(Zt[:, 0], Zt[:, 1], c=Y[sub], cmap=cmap, s=4); axes[1, j].set_title(f"t-SNE — {titles[k]}")
savefig("fig05_espaces_latents")

## 5. Méthodes d'ensemble (famille 2)

Trois mécanismes, appliqués aux **caractéristiques gelées** les plus performantes de la section 4 (après ACP à 64 dimensions) :
- **Bagging** : arbres bootstrap (`BaggingClassifier`) et forêt aléatoire, avec **erreur out-of-bag** ;
- **Boosting** : gradient boosting par histogrammes ;
- **Stacking** : régression logistique + forêt + boosting, méta-apprenant logistique entraîné sur des **prédictions hors-pli**, avec une validation croisée **groupée** par groupe de quasi-doublons (section 1.4) pour éviter qu'une même scène se retrouve des deux côtés.

Pourquoi des caractéristiques *gelées* et non celles du réseau fine-tuné : sur le train, ces dernières sont quasi parfaitement séparables (accuracy d'entraînement ≈ 99,9 %), ce qui fausserait les prédictions hors-pli (fuite d'information).

Réserve : l'erreur OOB reste optimiste car le rééchantillonnage se fait au niveau du recadrage et non de l'image.

In [ ]:
from sklearn.ensemble import BaggingClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold

cands = [k for k in ("imagenet", "dino") if k in EMB]
PROBE_NAME = {"imagenet": "ResNet50 ImageNet gelé + sonde linéaire", "dino": "DINO ResNet50 gelé + sonde linéaire"}
FEAT = max(cands, key=lambda k: RESULTS[PROBE_NAME[k]]["val"]["macro_f1"])
print("Caractéristiques retenues pour les ensembles :", FEAT)
pca_e = PCA(64, random_state=SEED).fit(EMB[FEAT][ID["train"]])
Ze = pca_e.transform(EMB[FEAT]).astype(np.float32)
Ztr, ytr = Ze[ID["train"]], Y[ID["train"]]; Zva, yva = Ze[ID["validation"]], Y[ID["validation"]]; Zte = Ze[ID["test"]]
grp = crops_df.filename.iloc[ID["train"]].map(lambda f: GROUP.get(f, hash(f) % (10**9))).values

ens_models = {
    "Bagging (arbres) + OOB": BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, n_jobs=-1, random_state=SEED),
    "Random Forest + OOB": RandomForestClassifier(300, oob_score=True, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "Gradient boosting (histogrammes)": HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, early_stopping=True, random_state=SEED),
}
ENS = {}
for name, clf in ens_models.items():
    t = time.time(); clf.fit(Ztr, ytr)
    pv = clf.predict_proba(Zva); ENS[name] = {"val": metrics(yva, pv.argmax(1), pv), "test_probs": clf.predict_proba(Zte)}
    if hasattr(clf, "oob_score_"): ENS[name]["oob"] = float(clf.oob_score_)
    print(f"{name:34s}", {k: round(v, 3) for k, v in ENS[name]["val"].items()}, "OOB:", round(ENS[name].get("oob", float('nan')), 3), f"({time.time()-t:.0f}s)")

In [ ]:
t = time.time()
folds = list(StratifiedGroupKFold(STACK_FOLDS, shuffle=True, random_state=SEED).split(Ztr, ytr, grp))
stack = StackingClassifier(
    estimators=[("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=0.5))),
                ("rf", RandomForestClassifier(150, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)),
                ("hgb", HistGradientBoostingClassifier(max_iter=60, early_stopping=True, random_state=SEED))],
    final_estimator=LogisticRegression(max_iter=500), cv=folds, stack_method="predict_proba", n_jobs=1)
stack.fit(Ztr, ytr)
pv = stack.predict_proba(Zva)
ENS["Stacking (hors-pli, plis groupés)"] = {"val": metrics(yva, pv.argmax(1), pv), "test_probs": stack.predict_proba(Zte)}
print("Stacking", {k: round(v, 3) for k, v in ENS["Stacking (hors-pli, plis groupés)"]["val"].items()}, f"({(time.time()-t)/60:.1f} min)")

ens_tab = pd.DataFrame({n: {**v["val"], "OOB": v.get("oob", np.nan)} for n, v in ENS.items()}).T
display(ens_tab.style.format("{:.3f}"))
BEST_ENS = max(ENS, key=lambda n: ENS[n]["val"]["macro_f1"])
RESULTS[f"Meilleur ensemble sur val. : {BEST_ENS}"] = {"val": ENS[BEST_ENS]["val"]}
TEST_PROBS[f"Meilleur ensemble sur val. : {BEST_ENS}"] = ENS[BEST_ENS]["test_probs"]
print("Meilleur ensemble (validation) :", BEST_ENS)

## 6. Modules spécialisés (famille 3)

### 6.1 Détection d'objets : YOLO11n à une classe
Pourquoi : une photo contient 1 à 38 avions de tailles très variées ; il faut d'abord les localiser. YOLO11n (2,6 M de paramètres, ≈ 4 ms/image) détecte « aircraft » indépendamment du modèle ; le classifieur n'a alors qu'à identifier le contenu de chaque boîte.

Les métriques de détection (précision, rappel, mAP) sont **recalculées** ici sur les 1 694 images de test avec les poids rechargés.

In [ ]:
from ultralytics import YOLO
yolo = YOLO(YOLO_W)
test_df = df[df.split == "test"]
if RUN_YOLO_MAP:
    root = OUT / "yolo_test"
    for d in ("images/test", "labels/test"):
        (root / d).mkdir(parents=True, exist_ok=True)
    for fn, g in tqdm(test_df.groupby("filename"), desc="export test YOLO"):
        dst = root / "images/test" / f"{fn}.jpg"
        if not dst.exists():
            shutil.copyfile(IMG_DIR / f"{fn}.jpg", dst)
        lines = [f"0 {(r.xmin+r.xmax)/2/r.width:.6f} {(r.ymin+r.ymax)/2/r.height:.6f} {r.box_w/r.width:.6f} {r.box_h/r.height:.6f}"
                 for r in g.itertuples()]
        (root / "labels/test" / f"{fn}.txt").write_text("\n".join(lines))
    (root / "data.yaml").write_text(f"path: {root}\ntrain: images/test\nval: images/test\nnc: 1\nnames:\n- aircraft\n")
    mm = yolo.val(data=str(root / "data.yaml"), split="val", imgsz=640, batch=16, conf=0.001, iou=0.6,
                  plots=False, verbose=False, device=0 if AMP else "cpu")
    DET = {"precision": float(mm.box.mp), "recall": float(mm.box.mr), "mAP50": float(mm.box.map50), "mAP50_95": float(mm.box.map)}
    print("Détection (test) :", {k: round(100 * v, 2) for k, v in DET.items()})
    shutil.rmtree(root / "images", ignore_errors=True)
else:
    DET = None

### 6.2 Pipeline complet et évaluation de bout en bout (protocoles B et C)
Chaque image de test est traitée : YOLO (conf ≥ 0,25, NMS IoU 0,5) → recadrage de chaque boîte → ResNet50 final. Appariement des boîtes prédites aux annotations (IoU ≥ 0,5, un-à-un glouton). Mesures :
- **B — conditionnelle** : avions des 34 classes détectés ;
- **C — bout en bout** : les avions manqués comptent comme erreurs ;
- **macro-F1 global** : fausses détections et avions hors des 34 classes rangés dans une classe « fond » ;
- **top-3** conditionnel.

In [ ]:
TFM_FT = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_imgs = sorted(test_df.filename.unique()); gt_by_img = {fn: g for fn, g in test_df.groupby("filename")}
BG, MISS = "__fond__", "__manqué__"
dets, gts = [], []
t0 = time.time()
with torch.inference_mode():
    for s in tqdm(range(0, len(test_imgs), 16), desc="pipeline"):
        chunk = test_imgs[s:s + 16]
        res = yolo.predict([str(IMG_DIR / f"{f}.jpg") for f in chunk], imgsz=640, conf=0.25, iou=0.5,
                           verbose=False, device=0 if AMP else "cpu")
        for fn, r in zip(chunk, res):
            img = r.orig_img; H_, W_ = img.shape[:2]
            boxes = r.boxes.xyxy.cpu().numpy(); dconf = r.boxes.conf.cpu().numpy()
            keep, tens = [], []
            for k, b in enumerate(boxes):
                x1, y1, x2, y2 = int(max(0, b[0])), int(max(0, b[1])), int(min(W_, b[2])), int(min(H_, b[3]))
                if x2 - x1 < 2 or y2 - y1 < 2: continue
                keep.append(k); tens.append(TFM_FT(Image.fromarray(np.ascontiguousarray(img[y1:y2, x1:x2, ::-1]))))
            pr = None
            if tens:
                pr = torch.softmax(ft(torch.stack(tens).to(DEVICE)).float(), 1).cpu().numpy()
            g = gt_by_img[fn]; gb = g[["xmin", "ymin", "xmax", "ymax"]].values; pb = boxes[keep] if keep else np.zeros((0, 4))
            m = greedy_match(gb, pb); matched_p = {j: (i, v) for i, (j, v) in m.items()}
            for jj in range(len(keep)):
                gi, iou = matched_p.get(jj, (None, 0.0))
                dets.append(dict(filename=fn, det_conf=float(dconf[keep[jj]]), probs=pr[jj], pred=int(pr[jj].argmax()),
                                 conf=float(pr[jj].max()), gt_class=(g.iloc[gi]["class"] if gi is not None else None),
                                 area=float((pb[jj, 2] - pb[jj, 0]) * (pb[jj, 3] - pb[jj, 1])), iou=iou))
            for gi, (_, row) in enumerate(g.iterrows()):
                gts.append(dict(filename=fn, cls=row["class"], supported=bool(row["class"] in CLASS_TO_IDX), matched=gi in m))
print(f"{(time.time()-t0)/60:.1f} min")
D, G = pd.DataFrame(dets), pd.DataFrame(gts)
D["kind"] = np.where(D.gt_class.isna(), "fp", np.where(D.gt_class.isin(CLASS_TO_IDX), "sup", "unsup"))
sup_m = D[D.kind == "sup"].copy(); sup_m["y"] = sup_m.gt_class.map(CLASS_TO_IDX)
P_sup = np.stack(sup_m.probs.values); correct = (sup_m.pred.values == sup_m.y.values)
n_sup_gt = int(G.supported.sum()); n_missed = int(((~G.matched) & G.supported).sum())
ev, yt, yp = [], [], []
for r in D.itertuples():
    if r.kind == "sup": yt.append(r.gt_class); yp.append(CLASSES[r.pred])
    else: yt.append(BG); yp.append(CLASSES[r.pred])
for r in G.itertuples():
    if r.supported and not r.matched: yt.append(r.cls); yp.append(MISS)
PIPE = {
    "images": len(test_imgs), "annotations_total": len(G), "annotations_34_classes": n_sup_gt,
    "detections": len(D), "matched_34_classes": len(sup_m), "missed_34_classes": n_missed,
    "unmatched_detections": int((D.kind == "fp").sum()), "detections_on_other_classes": int((D.kind == "unsup").sum()),
    "B_conditional": metrics(sup_m.y.values, sup_m.pred.values, P_sup),
    "C_end_to_end_accuracy": float(correct.sum() / n_sup_gt),
    "C_global_macro_f1_with_background": float(f1_score(yt, yp, labels=CLASSES, average="macro", zero_division=0)),
    "ece_conditional": ece_score(sup_m.conf.values, correct)}
print(json.dumps(PIPE, indent=2))

**Lecture.** L'écart entre B (conditionnelle) et C (bout en bout) mesure le coût des avions manqués par le détecteur ; la chute du macro-F1 global vient des avions **hors des 34 classes** et des fausses détections, qu'un classifieur fermé est forcé d'affecter à l'une des 34 classes.

#### Rejet des prédictions peu fiables (monde ouvert)
Sans réentraînement, un seuil sur la confiance du classifieur permet de refuser une prédiction (classe « autre »). On trace le compromis : part des avions corrects conservés, contre part des « intrus » (avions hors domaine et fausses détections) encore acceptés à tort.

In [ ]:
intr = D[D.kind != "sup"].conf.values
taus = np.linspace(0, 0.99, 100)
keep_correct = [(sup_m.conf.values[correct] >= t).mean() for t in taus]
acc_acc = [correct[sup_m.conf.values >= t].mean() if (sup_m.conf.values >= t).any() else np.nan for t in taus]
intr_acc = [(intr >= t).mean() for t in taus]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(taus, keep_correct, label="avions corrects conservés"); ax[0].plot(taus, intr_acc, label="intrus encore acceptés")
ax[0].set_xlabel("seuil de confiance τ"); ax[0].legend(); ax[0].set_title("Rejet par seuil de confiance")
ax[1].plot([(sup_m.conf.values >= t).mean() for t in taus], acc_acc); ax[1].set_xlabel("couverture"); ax[1].set_ylabel("accuracy sur les prédictions acceptées")
ax[1].set_title("Courbe risque-couverture")
savefig("fig06_rejet_monde_ouvert")
i90 = int(np.argmax(np.array(intr_acc) <= 0.5)); print(f"τ ≈ {taus[i90]:.2f} : {100*intr_acc[i90]:.0f} % des intrus acceptés, {100*keep_correct[i90]:.0f} % des corrects conservés")

### 6.3 Recherche d'images similaires (CBIR)
**Tâche.** Pour un recadrage requête (test), retrouver dans une galerie (recadrages du train) des avions *du même modèle*. Distance : similarité cosinus entre embeddings. Métriques : précision@1/5/10, taux de succès@5 et mAP (rang complet). Comparaison des représentations de la section 4 : le fine-tuning doit améliorer la qualité de la recherche.

In [ ]:
def retrieval_eval(Eg, yg, Eq, yq, ks=(1, 5, 10)):
    Gm = F.normalize(torch.tensor(Eg, dtype=torch.float32), dim=1).to(DEVICE)
    Qm = F.normalize(torch.tensor(Eq, dtype=torch.float32), dim=1).to(DEVICE)
    yg_t, yq_t = torch.tensor(yg).to(DEVICE), torch.tensor(yq).to(DEVICE)
    aps, prec, hit = [], {k: [] for k in ks}, []
    for s in range(0, len(Qm), 128):
        order = (Qm[s:s + 128] @ Gm.T).argsort(dim=1, descending=True)
        rel = (yg_t[order] == yq_t[s:s + 128, None]).float()
        cum = rel.cumsum(1); ranks = torch.arange(1, rel.shape[1] + 1, device=DEVICE)
        aps.append((((cum / ranks) * rel).sum(1) / rel.sum(1).clamp(min=1)).cpu())
        for k in ks: prec[k].append(rel[:, :k].mean(1).cpu())
        hit.append((rel[:, :5].sum(1) > 0).float().cpu())
    out = {f"P@{k}": float(torch.cat(prec[k]).mean()) for k in ks}
    out["Hit@5"] = float(torch.cat(hit).mean()); out["mAP"] = float(torch.cat(aps).mean())
    return out

CBIR = {titles[k]: retrieval_eval(EMB[k][ID["train"]], Y[ID["train"]], EMB[k][ID["test"]], Y[ID["test"]]) for k in names_emb}
cbir_tab = pd.DataFrame(CBIR).T; display(cbir_tab.style.format("{:.3f}"))
prior = (np.bincount(Y[ID["train"]], minlength=N_CLASSES) / len(ID["train"])) ** 2
print(f"Repère (tirage aléatoire) : précision@1 ≈ {prior.sum():.3f}")

In [ ]:
# Exemples de requêtes (embeddings du modèle fine-tuné)
Gm = F.normalize(torch.tensor(EMB["finetuned"][ID["train"]]), dim=1); rs = np.random.RandomState(7)
qs = rs.choice(ID["test"], 4, replace=False)
fig, axes = plt.subplots(4, 6, figsize=(15, 10.5))
for r, qi in enumerate(qs):
    q = F.normalize(torch.tensor(EMB["finetuned"][qi])[None], dim=1); top = (q @ Gm.T).squeeze(0).topk(5).indices.numpy()
    axes[r, 0].imshow(Image.open(PATHS[qi])); axes[r, 0].set_title(f"requête : {CLASSES[Y[qi]]}", color="blue", fontsize=9)
    for c, gi in enumerate(ID["train"][top], 1):
        axes[r, c].imshow(Image.open(PATHS[gi])); good = Y[gi] == Y[qi]
        axes[r, c].set_title(CLASSES[Y[gi]], color="green" if good else "red", fontsize=9)
for a in axes.ravel(): a.axis("off")
savefig("fig07_cbir_exemples")

## 7. Évaluation finale et analyse des erreurs

### 7.1 Comparaison de toutes les approches (protocole A) — lecture unique du test
Les prédictions de test ont été stockées sans être consultées. Pour les ensembles et la baseline classique, **seul le meilleur modèle selon la validation** est évalué sur le test.

In [ ]:
yte = Y[ID["test"]]
rows = []
for name, tp in TEST_PROBS.items():
    if isinstance(tp, tuple):
        mt = metrics(yte, tp[1])
    else:
        mt = metrics(yte, tp.argmax(1), tp)
    v = RESULTS[name]["val"]
    rows.append({"Modèle": name, "Val. acc.": v["accuracy"], "Val. macro-F1": v["macro_f1"], "Val. top-3": v.get("top3", np.nan),
                 "Test acc.": mt["accuracy"], "Test macro-F1": mt["macro_f1"], "Test top-3": mt.get("top3", np.nan)})
final_tab = pd.DataFrame(rows).set_index("Modèle")
display(final_tab.style.format("{:.3f}"))
print(f"Repère : tirage aléatoire ≈ {1/N_CLASSES:.3f} d'accuracy")
print("\nProtocole B (recadrages du détecteur, conditionnel) :", {k: round(v, 3) for k, v in PIPE["B_conditional"].items()})
print("Protocole C (bout en bout) : accuracy =", round(PIPE["C_end_to_end_accuracy"], 3), "· macro-F1 global =", round(PIPE["C_global_macro_f1_with_background"], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
final_tab[["Test acc.", "Test macro-F1"]].plot.barh(ax=ax, color=["#3FB8AF", "#E8A33D"])
ax.set_xlim(0, 1); ax.set_title("Test, protocole A : toutes familles de méthodes"); ax.invert_yaxis()
savefig("fig08_comparaison_modeles")

### 7.2 Matrice de confusion et classes difficiles (modèle final, protocole A)

In [ ]:
pf = FT_PROBS[ID["test"]]; pred_f = pf.argmax(1)
cm = confusion_matrix(yte, pred_f, labels=range(N_CLASSES)); cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.set_xticks(range(N_CLASSES), CLASSES, rotation=90); ax.set_yticks(range(N_CLASSES), CLASSES)
ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle"); ax.grid(False); fig.colorbar(im, fraction=0.03)
ax.set_title("Matrice de confusion normalisée — ResNet50 final, test (recadrages annotés)")
savefig("fig09_matrice_confusion")

err = cm.copy(); np.fill_diagonal(err, 0)
top_conf = pd.DataFrame([(CLASSES[i], CLASSES[j], int(err[i, j])) for i, j in zip(*np.nonzero(err))], columns=["réel", "prédit", "n"]).sort_values("n", ascending=False).head(12)
display(top_conf.reset_index(drop=True))
rep = pd.DataFrame(classification_report(yte, pred_f, labels=range(N_CLASSES), target_names=CLASSES, output_dict=True, zero_division=0)).T.iloc[:N_CLASSES]
rep["f1-score"].sort_values().plot.barh(figsize=(8, 9), color="#3FB8AF"); plt.title("F1 par classe (test, modèle final)"); savefig("fig10_f1_par_classe")

### 7.3 Calibration et erreurs très confiantes
Un classifieur est bien calibré si, parmi les prédictions faites avec 80 % de confiance, 80 % sont correctes.

In [ ]:
conf_f, corr_f = pf.max(1), (pred_f == yte)
ECE = ece_score(conf_f, corr_f)
edges = np.linspace(0, 1, 11); acc_b, mid = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf_f > lo) & (conf_f <= hi)
    acc_b.append(corr_f[m].mean() if m.any() else np.nan); mid.append((lo + hi) / 2)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].bar(mid, acc_b, width=0.09, color="#3FB8AF", label="accuracy observée"); ax[0].plot([0, 1], [0, 1], "k--", label="calibration parfaite")
ax[0].set_xlabel("confiance"); ax[0].set_title(f"Diagramme de fiabilité (ECE = {ECE:.3f})"); ax[0].legend()
ax[1].hist([conf_f[corr_f], conf_f[~corr_f]], bins=20, label=["corrects", "erreurs"], color=["#3FB8AF", "#E8A33D"]); ax[1].legend(); ax[1].set_xlabel("confiance")
savefig("fig11_calibration")
wrong = ~corr_f
print(f"{wrong.sum()} erreurs sur {len(yte)} · confiance moyenne des erreurs {conf_f[wrong].mean():.2f} · "
      f"≥ 90 % : {(conf_f[wrong] >= .9).sum()} · ≥ 95 % : {(conf_f[wrong] >= .95).sum()} · ≥ 99 % : {(conf_f[wrong] >= .99).sum()}")

### 7.4 Rôle de la taille de l'objet
Les erreurs sont-elles concentrées sur les petits avions ? Exactitude par quintile de surface de boîte (test, protocole A).

In [ ]:
te_df = crops_df.iloc[ID["test"]].copy(); te_df["correct"] = corr_f
te_df["quintile"] = pd.qcut(te_df.area, 5, labels=["Q1 (petits)", "Q2", "Q3", "Q4", "Q5 (grands)"])
size_tab = te_df.groupby("quintile").agg(accuracy=("correct", "mean"), surface_mediane=("area", "median"), n=("correct", "size"))
display(size_tab.style.format({"accuracy": "{:.3f}", "surface_mediane": "{:.0f}"}))
size_tab.accuracy.plot.bar(color="#E8A33D", figsize=(7, 3.8)); plt.ylim(0, 1); plt.ylabel("accuracy"); plt.title("Accuracy selon la taille (test)"); savefig("fig12_taille")

### 7.5 Impact des quasi-doublons sur le score de test
Accuracy sur les recadrages de test dont l'image a un quasi-doublon dans le train, contre les autres. Un écart important signalerait une fuite qui gonfle le score.

In [ ]:
if RUN_DEDUP:
    leak = te_df.filename.isin(LEAK_TEST).values
    print(f"Recadrages de test avec quasi-doublon dans le train : {leak.sum()} / {len(leak)}")
    if leak.sum() >= 10:
        print(f"accuracy (avec quasi-doublon) = {corr_f[leak].mean():.3f} · accuracy (sans) = {corr_f[~leak].mean():.3f} · "
              f"macro-F1 sans quasi-doublon = {f1_score(yte[~leak], pred_f[~leak], average='macro', zero_division=0):.3f}")
    else:
        print("Trop peu de quasi-doublons pour comparer : le score de test n'est pas gonflé par ce biais.")

### 7.6 Exemples d'erreurs les plus confiantes

In [ ]:
idx_err = np.where(wrong)[0]; worst = idx_err[np.argsort(-conf_f[idx_err])][:12]
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for a, k in zip(axes.ravel(), worst):
    a.imshow(Image.open(PATHS[ID["test"][k]])); a.axis("off")
    a.set_title(f"vrai : {CLASSES[yte[k]]}\nprédit : {CLASSES[pred_f[k]]} ({conf_f[k]*100:.0f} %)", fontsize=9, color="darkred")
savefig("fig13_erreurs_confiantes")

### 7.7 Monde ouvert : que fait le classifieur avec un avion qu'il ne connaît pas ?
Détections du pipeline (section 6.2) tombant sur des avions **hors des 34 classes** : le classifieur les range forcément dans l'une des 34.

In [ ]:
unsup = D[D.kind == "unsup"]
print(f"{len(unsup)} détections sur des avions hors des 34 classes ; confiance médiane du classifieur : {unsup.conf.median():.2f} ; "
      f"{(unsup.conf >= .9).mean()*100:.0f} % à ≥ 90 % de confiance")
display(unsup.pred.map(lambda i: CLASSES[i]).value_counts().head(8).rename("détections").to_frame())
display(unsup.gt_class.value_counts().head(8).rename("classe réelle (hors 34)").to_frame())

## 8. Limites (à discuter honnêtement à la soutenance)

1. **Classifieur fermé** : environ 41 % des avions du test appartiennent à d'autres classes ; la section 6.2 montre que le seuil de confiance n'élimine qu'une partie des intrus.
2. **Split non refait** : fourni par le jeu de données (niveau image). L'audit 1.4/7.5 quantifie la fuite par quasi-doublons mais ne l'élimine pas ; une re-découpe groupée imposerait de réentraîner YOLO et ResNet50.
3. **Trois protocoles non comparables entre eux** : A (boîtes exactes), B (boîtes détectées, avions manqués exclus) et C (bout en bout). Seul A permet de comparer les familles de méthodes.
4. **Baselines classiques entraînées sur un sous-échantillon** (`N_PILOT_CLASSIC`) à 128×128 : elles sous-estiment ce que des descripteurs classiques peuvent faire avec toutes les données.
5. **Ensembles sur caractéristiques gelées** et OOB optimiste (rééchantillonnage au niveau du recadrage).
6. **Gain du rééquilibrage confondu** avec 10 époques supplémentaires ; une expérience de contrôle (mêmes époques, échantillonnage uniforme) est nécessaire pour isoler l'effet.
7. **Entraînement non reproduit ici** : les poids viennent d'une exécution précédente (graines PyTorch non fixées pour les premiers régimes) ; seules les courbes d'apprentissage relevées sont affichées.
8. **Un seul tirage** : pas d'intervalle de confiance ni de répétition sur plusieurs graines ; les petites différences entre modèles ne sont pas significatives.
9. **Éthique** : données militaires, usage académique uniquement, pas de ciblage ni de renseignement opérationnel.

## 9. Sauvegarde des résultats

In [ ]:
from IPython.display import FileLink
import zipfile
def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and np.isnan(o): return None
    return o
final = {"env": ENV,
         "dataset": {"classes": N_CLASSES, "recadrages": {s: int(len(v)) for s, v in ID.items()}, "ratio_desequilibre_train": float(tr_counts.max() / tr_counts.min()),
                     "quasi_doublons_test": len(LEAK_TEST), "quasi_doublons_val": len(LEAK_VAL)},
         "baseline_classique": {"meilleur_modele": BEST_CLASSIC["model"], "table_val": classic_table.to_dict("records")},
         "comparaison_protocole_A": final_tab.reset_index().to_dict("records"),
         "ensembles_val": {n: {**v["val"], "oob": v.get("oob")} for n, v in ENS.items()},
         "detection_yolo_test": DET, "pipeline_B_C": {k: v for k, v in PIPE.items()},
         "cbir": CBIR, "calibration_ECE_protocole_A": ECE}
(OUT / "final_results.json").write_text(json.dumps(clean(final), indent=2, ensure_ascii=False))
zp = OUT / "resultats_notebook.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / "final_results.json", "final_results.json")
    for p in sorted(FIG.glob("*.png")): zf.write(p, f"figures/{p.name}")
print(round(zp.stat().st_size / 2**20, 1), "Mo"); display(FileLink(str(zp)))

## 10. Conclusion

Les familles de méthodes du cours sont couvertes et comparées avec un protocole commun : descripteurs classiques (HOG/LBP + SVM, KNN, forêt), représentations apprises (ImageNet gelé, DINO, autoencodeur, fine-tuning), méthodes d'ensemble (bagging/OOB, boosting, stacking hors-pli) et modules spécialisés (détection YOLO11n, CBIR). Les tableaux des sections 5, 6 et 7 donnent les chiffres de **cette exécution** ; reportez-les tels quels dans le rapport et les slides.

Messages à retenir : (1) l'adaptation au domaine par fine-tuning apporte l'essentiel du gain ; (2) la performance de bout en bout est plus faible que la performance conditionnelle (avions manqués, intrus) ; (3) un classifieur fermé est mal calibré face à des avions inconnus ; (4) les limites de la section 8 doivent accompagner chaque chiffre annoncé.

TEST

In [ ]:
import torch, torch.nn as nn, glob
from torchvision import models, transforms
from PIL import Image
from ultralytics import YOLO
import numpy as np

# ---------- Chemins (adaptés à ton environnement) ----------
YOLO_W   = glob.glob("/kaggle/input/**/yolo11n_best.pt", recursive=True)[0]
RESNET_W = glob.glob("/kaggle/input/**/resnet50_balanced_best.pth", recursive=True)[0]
print("YOLO   :", YOLO_W)
print("ResNet :", RESNET_W)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

# ---------- 1. Classifieur ResNet50 ----------
ck = torch.load(RESNET_W, map_location="cpu", weights_only=False)
print("Clés du checkpoint :", list(ck.keys()))   # utile pour vérifier

CLASS_TO_IDX = ck["class_to_idx"]
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
N_CLASSES = len(CLASS_TO_IDX)
print("Nombre de classes :", N_CLASSES)

clf = models.resnet50(weights=None)
clf.fc = nn.Linear(clf.fc.in_features, N_CLASSES)

# Le state_dict peut être sous différentes clés selon comment tu l'as sauvé
if "model_state_dict" in ck:
    state = ck["model_state_dict"]
elif "state_dict" in ck:
    state = ck["state_dict"]
else:
    state = ck
clf.load_state_dict(state)
clf.to(DEVICE).eval()

# ---------- 2. Prétraitement (vérifie avec ton entraînement !) ----------
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ---------- 3. Détecteur YOLO ----------
det = YOLO(YOLO_W)

# ---------- 4. Fonction pipeline ----------
@torch.no_grad()
def predict_image(path, conf_yolo=0.25, conf_clf=0.5, margin_unknown=0.15, iou=0.45):
    img = Image.open(path).convert("RGB")
    W, H = img.size
    res = det.predict(img, conf=conf_yolo, iou=iou, verbose=False)[0]
    out = []
    for box in res.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        crop = img.crop((x1, y1, x2, y2))
        if crop.width < 8 or crop.height < 8:
            continue
        x = tf(crop).unsqueeze(0).to(DEVICE)
        logits = clf(x)
        probs = torch.softmax(logits, 1)[0].cpu().numpy()
        k = int(probs.argmax())
        top1, top2 = probs[k], np.partition(probs, -2)[-2]
        if top1 < conf_clf or (top1 - top2) < margin_unknown:
            label = "INCONNU / hors 34 classes"
        else:
            label = IDX_TO_CLASS[k]
        out.append({
            "box": (x1, y1, x2, y2),
            "yolo_conf": float(box.conf[0]),
            "label": label,
            "clf_conf": float(top1),
            "top3": [(IDX_TO_CLASS[i], float(probs[i]))
                     for i in np.argsort(-probs)[:3]],
        })
    return out

print("predict_image prête ✅")

In [ ]:
for r in predict_image("/kaggle/input/datasets/bichotasomibola/mon-image/tu-160.webp"):
    print(f'{r["label"]:30s} | clf={r["clf_conf"]:.3f} | yolo={r["yolo_conf"]:.3f} | box={r["box"]}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def show_predictions(path, conf_yolo=0.25, conf_clf=0.5, margin_unknown=0.15, iou=0.45):
    results = predict_image(path, conf_yolo=conf_yolo, conf_clf=conf_clf,
                            margin_unknown=margin_unknown, iou=iou)
    img = Image.open(path).convert("RGB")

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img)
    ax.axis("off")

    if not results:
        ax.set_title("Aucun avion détecté (essaie conf_yolo=0.15)", color="orange")
    for r in results:
        x1, y1, x2, y2 = r["box"]
        unknown = r["label"].startswith("INCONNU")
        color = "red" if unknown else "lime"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, color=color, lw=2))
        ax.text(x1, max(y1 - 6, 10),
                f'{r["label"]}  clf={r["clf_conf"]:.2f}  yolo={r["yolo_conf"]:.2f}',
                color="white", fontsize=10, weight="bold",
                bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor="none"))

    plt.tight_layout()
    plt.show()
    return results

# ---------- Utilisation ----------
res = show_predictions("/kaggle/input/datasets/bichotasomibola/mon-image/tu-160.webp")

# Détail texte (top-3) pour chaque avion
for r in res:
    print(f'{r["label"]:30s} | clf={r["clf_conf"]:.3f} | yolo={r["yolo_conf"]:.3f} | box={r["box"]}')
    print("   top-3 :", [(c, round(p, 3)) for c, p in r["top3"]])